# v03 — Comparação de modelos e seleção de variáveis

Duas perguntas, mesmos folds:

1. Modelos de árvore (`HistGradientBoostingClassifier`, `RandomForestClassifier`) superam a logística?
2. A logística precisa de todas as 17 variáveis, ou o Elo já concentra a informação?

In [1]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

DATA = Path('../data')
SUB_DIR = Path('../submissions')
SUB_DIR.mkdir(exist_ok=True)
LOG_PATH = Path('../experimentos.csv')

CLASS_ORDER = ['A', 'D', 'H']
RANDOM_STATE = 42
FOLDS = [2016, 2017, 2018, 2019, 2020, 2021]  # walk-forward: treino = temporadas anteriores

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(train.shape, test.shape)

(3419, 26) (1725, 22)


In [2]:
NUM_FEATURES = ['elo_home', 'elo_away', 'elo_diff', 'form_pts_home', 'form_pts_away',
                'form_gf_home', 'form_ga_home', 'form_gf_away', 'form_ga_away',
                'home_form_as_host', 'away_form_as_visitor', 'rest_days_home', 'rest_days_away',
                'season_ppg_home', 'season_ppg_away', 'round_n', 'h2h_home_pts']


def add_features(df):
    """Features derivadas usando apenas colunas da propria linha."""
    out = df.copy()
    out['venue_form_diff'] = out['home_form_as_host'] - out['away_form_as_visitor']
    out['form_pts_diff'] = out['form_pts_home'] - out['form_pts_away']
    out['gd_diff'] = (out['form_gf_home'] - out['form_ga_home']) - (out['form_gf_away'] - out['form_ga_away'])
    return out


train = add_features(train)
test = add_features(test)

In [3]:
def make_logreg(num_cols, cat_cols=(), C=1.0):
    transformers = [('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                      ('scale', StandardScaler())]), list(num_cols))]
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), list(cat_cols)))
    return Pipeline([('prep', ColumnTransformer(transformers)),
                     ('model', LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE))])


def predict_ordered(model, X):
    """Alinha predict_proba a ['A', 'D', 'H'] pelas classes do modelo."""
    raw = pd.DataFrame(model.predict_proba(X), columns=model.named_steps['model'].classes_)
    return raw.reindex(columns=CLASS_ORDER, fill_value=0.0).values


def walk_forward(build, cols):
    """Previsoes out-of-fold por temporada; o modelo e reconstruido do zero a cada fold."""
    preds, ys = {}, {}
    for season in FOLDS:
        tr = train[train['Season'] < season]
        va = train[train['Season'] == season]
        model = build()
        model.fit(tr[cols], tr['Res'])
        preds[season] = predict_ordered(model, va[cols])
        ys[season] = va['Res'].values
    return preds, ys


def summarize(preds, ys, name):
    rows = [{'Season': s, 'n': len(ys[s]), 'LogLoss': log_loss(ys[s], preds[s], labels=CLASS_ORDER)}
            for s in FOLDS]
    df = pd.DataFrame(rows)
    mean = np.average(df['LogLoss'], weights=df['n'])
    print(f'{name}: media ponderada = {mean:.4f} | desvio entre folds = {df.LogLoss.std():.4f}')
    return df, mean

In [4]:
def save_submission(proba, path):
    sub = pd.DataFrame(proba, columns=CLASS_ORDER)
    sub.insert(0, 'Id', test['Id'].values)
    assert list(sub.columns) == ['Id', 'A', 'D', 'H']
    assert len(sub) == len(test) and (sub['Id'].values == test['Id'].values).all()
    vals = sub[CLASS_ORDER].values
    assert np.isfinite(vals).all() and (vals > 0).all()
    assert np.allclose(vals.sum(axis=1), 1.0, atol=1e-9)
    sub.to_csv(path, index=False)
    print('salvo em', path, sub.shape)
    return sub


def log_experiment(version, change, features, model_desc, params, folds_df, mean, decision, csv_path):
    row = {'versao': version, 'data': date.today().isoformat(), 'mudanca': change,
           'features': features, 'modelo': model_desc, 'parametros': params,
           'folds': ','.join(map(str, FOLDS)),
           'logloss_por_fold': ';'.join(f'{s}:{v:.4f}' for s, v in zip(folds_df.Season, folds_df.LogLoss)),
           'logloss_medio': round(mean, 4), 'desvio': round(folds_df.LogLoss.std(), 4),
           'decisao': decision, 'csv': csv_path}
    log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame(columns=list(row.keys()))
    log = log[log['versao'] != version]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    return log

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

def build_hgb():
    return Pipeline([('model', HistGradientBoostingClassifier(max_depth=2, learning_rate=0.03, max_iter=150,
                                                              l2_regularization=1.0, min_samples_leaf=80,
                                                              random_state=RANDOM_STATE))])

def build_rf():
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('model', RandomForestClassifier(n_estimators=500, min_samples_leaf=60, max_features=0.5,
                                                      random_state=RANDOM_STATE, n_jobs=-1))])

out = {}
preds, ys = walk_forward(lambda: make_logreg(NUM_FEATURES, C=0.003), NUM_FEATURES)
out['logreg 17 (v02)'] = summarize(preds, ys, 'logreg 17 (v02)')
preds, _ = walk_forward(build_hgb, NUM_FEATURES)
out['hgb 17'] = summarize(preds, ys, 'hgb 17')
preds, _ = walk_forward(build_rf, NUM_FEATURES)
out['rf 17'] = summarize(preds, ys, 'rf 17')
for C in [0.3, 0.1, 0.03, 0.01]:
    preds, _ = walk_forward(lambda: make_logreg(['elo_diff'], C=C), ['elo_diff'])
    out[f'logreg elo_diff C={C}'] = summarize(preds, ys, f'logreg elo_diff C={C}')

logreg 17 (v02): media ponderada = 1.0273 | desvio entre folds = 0.0403


hgb 17: media ponderada = 1.0408 | desvio entre folds = 0.0428


rf 17: media ponderada = 1.0289 | desvio entre folds = 0.0368
logreg elo_diff C=0.3: media ponderada = 1.0228 | desvio entre folds = 0.0416


logreg elo_diff C=0.1: media ponderada = 1.0228 | desvio entre folds = 0.0414
logreg elo_diff C=0.03: media ponderada = 1.0229 | desvio entre folds = 0.0408


logreg elo_diff C=0.01: media ponderada = 1.0234 | desvio entre folds = 0.0392


In [6]:
tab = pd.DataFrame({k: v[0].LogLoss.values for k, v in out.items()}, index=FOLDS).T
tab['media'] = [v[1] for v in out.values()]
tab.round(4).sort_values('media')

,2016,2017,2018,2019,2020,2021,media
logreg elo_diff C=0.3,0.9969,1.0878,0.9798,0.9891,1.0476,1.0356,1.0228
logreg elo_diff C=0.1,0.9969,1.0875,0.9800,0.9894,1.0475,1.0356,1.0228
logreg elo_diff C=0.03,0.9969,1.0864,0.9806,0.9904,1.0474,1.0356,1.0229
logreg elo_diff C=0.01,0.9974,1.0842,0.9823,0.9933,1.0473,1.0360,1.0234
logreg 17 (v02),1.0017,1.0924,0.9839,1.0008,1.0521,1.0328,1.0273
rf 17,1.0082,1.0879,0.9873,1.0029,1.0463,1.0406,1.0289
hgb 17,1.0323,1.1183,0.9957,1.0090,1.0450,1.0447,1.0408


As árvores ficam piores que a logística (overfit com ~2.000–3.000 partidas). O resultado mais interessante é que a logística só com `elo_diff` supera a versão com 17 variáveis: o Elo já resume forma e força, e as demais acrescentam mais ruído que sinal. A v03 usa só `elo_diff`.

In [7]:
key = 'logreg elo_diff C=0.1'
cols = ['elo_diff']
final = make_logreg(cols, C=0.1).fit(train[cols], train['Res'])
sub = save_submission(predict_ordered(final, test[cols]), SUB_DIR / 'submission_v03.csv')
log_experiment('v03', 'selecao: so elo_diff (arvores rejeitadas)', 'elo_diff', 'LogisticRegression', 'C=0.1',
               out[key][0], out[key][1], 'promovido', 'submissions/submission_v03.csv')

salvo em ..\submissions\submission_v03.csv (1725, 4)


,versao,data,mudanca,features,modelo,parametros,folds,logloss_por_fold,logloss_medio,desvio,decisao,csv
0,v01,2026-09-28,baseline logreg,17 numericas,LogisticRegression,C=1,"2016,2017,2018,2019,2020,2021",2016:1.0119;2017:1.1087;2018:0.9857;2019:0.997...,1.0337,0.0455,campeao inicial,submissions/submission_v01.csv
1,v02,2026-09-28,ajuste de C,17 numericas,LogisticRegression,C=0.003,"2016,2017,2018,2019,2020,2021",2016:1.0017;2017:1.0924;2018:0.9839;2019:1.000...,1.0273,0.0403,promovido,submissions/submission_v02.csv
2,v03,2026-09-28,selecao: so elo_diff (arvores rejeitadas),elo_diff,LogisticRegression,C=0.1,"2016,2017,2018,2019,2020,2021",2016:0.9969;2017:1.0875;2018:0.9800;2019:0.989...,1.0228,0.0414,promovido,submissions/submission_v03.csv
